# Make data for BioGeoBEARS

## Loading packages

In [1]:
library("tidygraph")
library("igraph")
library("ggraph")
library("tidyverse")


Attachement du package : ‘tidygraph’


L'objet suivant est masqué depuis ‘package:stats’:

    filter



Attachement du package : ‘igraph’


L'objet suivant est masqué depuis ‘package:tidygraph’:

    groups


Les objets suivants sont masqués depuis ‘package:stats’:

    decompose, spectrum


L'objet suivant est masqué depuis ‘package:base’:

    union


Le chargement a nécessité le package : ggplot2

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ lubridate 1.9.3     ✔ tibble    3.2.1
✔ purrr     1.0.2     ✔ tidyr     1.3.1
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ lubridate::%--%()      masks igraph::%--%()
✖ dplyr::as_data_frame() masks tibble::as_data_frame(), igraph::as_data_frame()
✖ purrr::compose()       masks igraph::compose()
✖ tidyr::crossing()      masks igraph::crossing()
✖ dplyr::filter()        masks tidygraph::filte

## Loading data

In [2]:
table <- read.table("Time_periods.tsv", sep ="\t", header = TRUE)

## Creating function to clean and save data

In [3]:
make_table_BioGeoBEARS <- function(time_periods_table, prefix){
    temp_length <- (ncol(time_periods_table) - 2)
    table_adjacency <- c()
    table_dispersal <- c()
    temp_mat_adj <- matrix(0, temp_length, temp_length)
    for(i in 1:nrow(time_periods_table)){
        temp_mat_adj <- matrix(0, temp_length, temp_length)
        temp_period_table <- time_periods_table[i, c(3:ncol(time_periods_table))]
        from <- c()
        to <- c()
        
        for(j in 1:temp_length){
            if(temp_period_table[j] != "0"){
                temp_mat_adj[j,eval(parse(text = temp_period_table[j]))] <- 1
                from <- c(from, rep(j,length(eval(parse(text = temp_period_table[j])))))
                to <- c(to, eval(parse(text = temp_period_table[j])))
            }
        }
        
        nodes <- tibble(id = 1:temp_length)
        edges <- tibble(from = from, to = to)
        
        temp_data_from <- edges[edges[,1] != edges[,2],]

        temp_data_to_1 <- edges[edges[,1] != edges[,2],]

        temp_data_to_2 <- edges[edges[,1] != edges[,2],]

        colnames(temp_data_to_1) <- c("to", "to_2")

        colnames(temp_data_to_2) <- c("to_2", "to_3")

        temp_data_03 <- merge(temp_data_from, temp_data_to_1, by = "to")
        temp_data_03 <- temp_data_03[temp_data_03[,2] != temp_data_03[,3],]

        temp_data_04 <- merge(temp_data_03, temp_data_to_1, by = "to_2")
        temp_data_04 <- temp_data_04[temp_data_04[,3] != temp_data_04[,4],]

        temp_mat_dispersal <- matrix(0.001, 7, 7)
        
        temp_data_direct <- as.matrix(edges)
        
        for(k in 1:nrow(temp_data_direct)){
            if(temp_data_direct[k,1] == temp_data_direct[k,2]){
                temp_mat_dispersal[temp_data_direct[k,1], temp_data_direct[k,2]] <- 1 
            }
            if(temp_data_direct[k,1] != temp_data_direct[k,2]){
                temp_mat_dispersal[temp_data_direct[k,1], temp_data_direct[k,2]] <- 0.5
            }
        
        }

        for(k in 1:nrow(temp_data_03)){
            if(temp_mat_dispersal[temp_data_03[k,2], temp_data_03[k,3]] == 0.0000001){
                temp_mat_dispersal[temp_data_03[k,2], temp_data_03[k,3]] <- 0.25
            }
        }

        for(k in 1:nrow(temp_data_04)){
            if(temp_mat_dispersal[temp_data_04[k,3], temp_data_04[k,4]] == 0.0000001){
                temp_mat_dispersal[temp_data_04[k,3], temp_data_04[k,4]] <- 0.125
            }
        }
        
        
        my_graph <- tbl_graph(nodes = nodes, edges = edges, directed = FALSE)
        
        connectivity_graph <-ggraph(my_graph, layout = "stress") +
            geom_edge_link() +
            geom_node_point(size = 10, colour = "antiquewhite") +
            geom_node_text(aes(label = id)) +
            theme_graph()
        
        ggsave(paste("Connection/connection_grap_", as.character(time_periods_table[i,2]), ".pdf",sep = ""))
        
        table_adjacency <- rbind(table_adjacency, LETTERS[1:7], temp_mat_adj, matrix(data=" ", ncol=7, nrow=1))
        
        table_dispersal <- rbind(table_dispersal, LETTERS[1:7], temp_mat_dispersal, matrix(data=" ", ncol=7, nrow=1))
    }
    table_adjacency <- rbind(table_adjacency, cbind("END", matrix(data = " ", ncol=6, nrow=1)))
    table_dispersal <- rbind(table_dispersal, cbind("END", matrix(data = " ", ncol=6, nrow=1)))
    write.table(time_periods_table[,2], paste(prefix, "_time_period.txt", sep = ""), sep ="\t", row.names = FALSE, col.names = FALSE, quote = FALSE)
    write.table(table_adjacency, paste(prefix, "_area_matrix.txt", sep = ""), sep ="\t", row.names = FALSE, col.names = FALSE, quote = FALSE)
    write.table(table_dispersal, paste(prefix, "_dispersal_matrix.txt", sep = ""), sep ="\t", row.names = FALSE, col.names = FALSE, quote = FALSE)
}

## Execute function

In [4]:
data <- make_table_BioGeoBEARS(table, "7_area")

Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
